# NER (Named Entity Recognition) ဆိုတာဘာလဲ?

NER က sentence ထဲက အရေးကြီးတဲ့ entities တွေကို ရှာဖွေတဲ့ NLP Task တစ်ခုပါ။

```
Elon Musk founded SpaceX in California.
```

```
| Word       | Label |
| ---------- | ----- |
| Elon       | B-PER |
| Musk       | I-PER |
| founded    | O     |
| SpaceX     | B-ORG |
| in         | O     |
| California | B-LOC |
```

Meaning
- PER = Person
- ORG = Organization
- LOC = Location
- O = Not Entity

# BERT NER Pipeline

```
Dataset
  ↓
Tokens + Labels
  ↓
BERT Tokenizer
  ↓
Label Alignment
  ↓
Input IDs
  ↓
BERT Encoder
  ↓
Token Classification Head
  ↓
Predicted Labels
  ↓
Precision / Recall / F1
  ↓
Save Model
  ↓
Inference
  ↓
Gradio App
```

NER Project မှာ အရေးကြီးဆုံး concept ၃ ခု က

* Tokenization
* Label Alignment
* Token Classification

ဖြစ်ပြီး၊ အထူးသဖြင့် Label Alignment ကို မနားလည်ရင် BERT-based NER ကို တကယ်နားလည်တယ်လို့ မပြောနိုင်ပါဘူး။

ဒါက word-level labels ကို BERT token-level labels အဖြစ် ပြောင်းပေးတဲ့ bridge step ဖြစ်လို့ပါ။

# Load Dataset (HuggingFace API)

In [3]:
from datasets import load_dataset

dataset = load_dataset("unimelb-nlp/wikiann", "en")

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/158k [00:00<?, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

```

| Field    | Meaning                           |
| -------- | --------------------------------- |
| tokens   | words in sentence                 |
| ner_tags | label IDs (BIO format)            |
| langs    | language code                     |
| spans    | entity spans (optional structure) |

```


In [4]:
# label names

label_names = dataset["train"].features["ner_tags"].feature.names
print(label_names)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


---

# Data Preprocessing

## Tokenization

- BERT က words မစားဘူး။

- BERT က tokens စားတယ်။

Example: ```Washington``` ကို BERT Tokenizer က ```Wash ##ington``` ဒီလိုပြောင်းမှာ။

**ဒီနေရာမှာ ပြဿနာဖြစ်လာတယ်**

Original label: ```Washington -> B-LOC``` က ဒီလိုတပ်မှာ ဒါမဲ့ Tokenized ထားတာကြတော့
```
Wash
##ington
```

ဘယ် label ပေးမလဲ?

## Label Alignment (အရမ်းအရေးကြီး)

NER project ရဲ့ အခက်ဆုံးနဲ့ အရေးကြီးဆုံး step ဖြစ်တယ်။

```
Original

Elon Musk founded SpaceX
```
```
Labels

B-PER
I-PER
O
B-ORG
```

---

```
BERT Tokenization

Elon
Mu
##sk
founded
Space
##X
```
```
Now labels?

Elon      -> B-PER
Mu        -> I-PER
##sk      -> I-PER

founded   -> O

Space     -> B-ORG
##X       -> I-ORG
```

Model က token level prediction လုပ်တာဖြစ်လို့

token တစ်ခုချင်းစီ label ရှိရမယ်။

ဒါကို Label Alignment လို့ခေါ်တယ်။

### Create Tokenized Dataset (BERT)

Your dataset is already good — you just need token alignment.

⚙️ Tokenization + Alignment Code (FIXED FOR YOUR DATASET)

In [5]:
from transformers import AutoTokenizer

model_name = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):

    # Tokenize
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    # Label Alignment
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids: # word_ids
            if word_idx is None:
                label_ids.append(-100) # Special tokens, no label, so pytorch loss function ignores -100
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [6]:
tokenized_dataset

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})

BERT စားနိုင်တဲ့ format ဖြစ်သွားပြီ။

---

# Model Training & Evaluation

## Build BERT Model

In [7]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_names)
)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

## Data Collator (IMPORTANT)

NER needs padding alignment. Why?

Sentence length မတူဘူး။
```
I love NLP = 3 tokens

Elon Musk founded SpaceX = 4 tokens

Batch ထဲထည့်ရင် shape တူရမယ်။
```
```
Padding

I love NLP [PAD]
```

ဒါကို automatic လုပ်ပေးတာ DataCollatorForTokenClassification ဖြစ်တယ်။

In [14]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

## Metrics (F1 score)

In [9]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [10]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=82ba6677197190a0b0d7fa9a638c2224b1dd0d8ab0cb593fa26ef95fd479770c
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [11]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_preds = []
    true_labels = []

    for pred, label in zip(predictions, labels):

        pred_tokens = []
        label_tokens = []

        for p_i, l_i in zip(pred, label):

            # ignore padding tokens
            if l_i != -100:

                pred_tokens.append(label_names[p_i])
                label_tokens.append(label_names[l_i])

        true_preds.append(pred_tokens)
        true_labels.append(label_tokens)

    results = seqeval.compute(
        predictions=true_preds,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],

        # ⭐ extra (recommended for portfolio)
        "accuracy": results.get("overall_accuracy", 0.0)
    }

## Training Arguments (Main Step)

Setup ပါပဲ။

In [12]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./ner_model",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=1,
    weight_decay=0.01,

    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

## Train the model

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.310406,0.274436,0.811323,0.829464,0.820293,0.916315


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1250, training_loss=0.3861397994995117, metrics={'train_runtime': 169.6467, 'train_samples_per_second': 117.892, 'train_steps_per_second': 7.368, 'total_flos': 320576618286144.0, 'train_loss': 0.3861397994995117, 'epoch': 1.0})

## Evaluate the model

In [18]:
trainer.evaluate(tokenized_dataset["test"])

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.310406,0.260128,1,0.817782,0.837528,0.827537,0.920564


{'eval_loss': 0.2601279020309448,
 'eval_precision': 0.8177822177822178,
 'eval_recall': 0.8375281358706773,
 'eval_f1': 0.8275374039627983,
 'eval_accuracy': 0.9205642688162622}

## Save Model

In [19]:
trainer.save_model("./ner_model")
tokenizer.save_pretrained("./ner_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./ner_model/tokenizer_config.json', './ner_model/tokenizer.json')

---

# Model Deployment

In [ ]:
# !pip install gradio transformers torch

## Load the model

In [20]:
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_path = "./ner_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

label_names = model.config.id2label

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [21]:
label_names = [
    "O",
    "B-PER",
    "I-PER",
    "B-ORG",
    "I-ORG",
    "B-LOC",
    "I-LOC"
]

label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

## Inference

Subword Handling?
```
BERT

Washington

↓

Wash
##ington
```

UI မှာ
```
Wash -> B-LOC
##ington -> I-LOC
```
ဆိုရင် မလှဘူး။ ဒါကြောင့်

```python
if word.startswith("##"):
```

နဲ့ ```Wash + ington``` ပေါင်းပြီး ```Washington``` ပြန်လုပ်တယ်။

In [22]:
def predict_entities(text):

    tokens = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**tokens)

    predictions = torch.argmax(outputs.logits, dim=2)[0]

    words = tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])

    results = []

    for word, pred in zip(words, predictions):

        label = model.config.id2label[pred.item()]

        if word not in ["[CLS]", "[SEP]", "[PAD]"]:
            results.append((word, label))

    return results

In [23]:
def format_output(text):

    tokens = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**tokens)

    predictions = torch.argmax(outputs.logits, dim=2)[0]

    words = tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])

    result = []
    current_word = ""
    current_label = None

    for word, pred in zip(words, predictions):

        label = label_names[pred.item()]

        if word in ["[CLS]", "[SEP]"]:
            continue

        # subword handling
        if word.startswith("##"):
            current_word += word[2:]
        else:
            if current_word:
                result.append(f"{current_word} → {current_label}")

            current_word = word
            current_label = label

    if current_word:
        result.append(f"{current_word} → {current_label}")

    return " | ".join(result)

## Gradio APP

In [26]:
with gr.Blocks() as demo:

    gr.Markdown("# Named Entity Recognition (NER) Demo")

    input_box = gr.Textbox(
        label="Enter text",
        placeholder="Type a sentence like: Elon Musk founded SpaceX in California"
    )

    output_box = gr.Textbox(label="NER Output")

    button = gr.Button("Extract Entities")

    button.click(fn=format_output, inputs=input_box, outputs=output_box)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3b47ef7161c79b570c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Report

> The fine-tuned BERT-based Named Entity Recognition model achieved an F1-score of 82.75%, with a precision of 81.78% and recall of 83.75% on the test set. The model obtained a token-level accuracy of 92.06% and an evaluation loss of 0.26, demonstrating strong entity extraction performance and good generalization capability on unseen data.

---

---

---

---